# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
ruta = "/Volumes/workspace/default/lab2/ventas_export.csv"

bronce = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ruta)
)

bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
CREATE OR REPLACE TEMP VIEW dim_tienda AS
SELECT * FROM VALUES
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
AS t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

filas_oro_roto
59


## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# Escriban aquí el flujo. No usen el CSV editado.
# spark.sql("create or replace temp view oro_fact_ventas as ...")
# spark.sql("create or replace temp view cuarentena_ventas as ...")
# Interpreté la coma como separador decimal.
# Cuando había coma, eliminé los puntos de miles.
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_limpias AS
SELECT *,
    TRY_CAST(tienda_id AS INT) AS tienda_num,
    TRY_CAST(
        CASE
            WHEN monto LIKE '%,%'
            THEN REPLACE(REPLACE(monto, '.', ''), ',', '.')
            ELSE monto
        END AS DECIMAL(18,2)
    ) AS monto_num
FROM bronce_ventas
""")

# Detecté los IDs repetidos con datos diferentes.
# Excluí canal de la comparación porque no pertenece al contrato.
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_conflictivas AS
SELECT venta_id
FROM (
    SELECT DISTINCT
        venta_id, fecha, tienda_num, producto_id,
        cantidad, monto_num, medio_pago
    FROM ventas_limpias
)
GROUP BY venta_id
HAVING COUNT(*) > 1
""")

# Asigné un motivo a los registros que no podían entrar.
# Numeré las filas de cada venta para conservar solo una copia.
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_revisadas AS
SELECT *,
    ROW_NUMBER() OVER (
        PARTITION BY venta_id
        ORDER BY fecha, monto, canal
    ) AS fila,
    CASE
        WHEN venta_id IS NULL OR TRIM(venta_id) = ''
            THEN 'Venta sin identificador'
        WHEN tienda_num IS NULL OR tienda_num NOT IN (1, 2, 3)
            THEN 'Tienda no permitida'
        WHEN monto_num IS NULL
            THEN 'Monto vacío o no numérico'
        WHEN venta_id IN (
            SELECT venta_id FROM ventas_conflictivas
        )
            THEN 'Venta repetida con datos diferentes'
    END AS problema
FROM ventas_limpias
""")

# Conservé una fila válida por venta y las columnas del contrato.
# Reemplacé la vista en cada ejecución para evitar acumular filas.
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT
    venta_id, fecha, tienda_num AS tienda_id,
    producto_id, cantidad, monto_num AS monto, medio_pago
FROM ventas_revisadas
WHERE problema IS NULL AND fila = 1
""")

# Conservé los valores originales de lo apartado y su motivo.
spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT
    venta_id, fecha, tienda_id, producto_id,
    cantidad, monto, medio_pago, canal,
    COALESCE(problema, 'Duplicado de una venta ya incluida') AS motivo
FROM ventas_revisadas
WHERE problema IS NOT NULL OR fila > 1
""")

print("Filas en oro:", spark.table("oro_fact_ventas").count())
print("Filas en cuarentena:", spark.table("cuarentena_ventas").count())

display(spark.table("cuarentena_ventas"))

Filas en oro: 26
Filas en cuarentena: 33


venta_id,fecha,tienda_id,producto_id,cantidad,monto,medio_pago,canal,motivo
1042,2026-10-02,1,501,1,18.50,efectivo,tienda,Duplicado de una venta ya incluida
1101,2026-10-03,1,510,1,null,efectivo,tienda,Monto vacío o no numérico
1202,2026-10-03,2,521,1,N/D,tarjeta,tienda,Monto vacío o no numérico
1301,2026-10-04,99,530,1,11.00,efectivo,whatsapp,Tienda no permitida
1303,2026-10-04,4,532,1,20.00,efectivo,tienda,Tienda no permitida
2001,2026-10-01,1,501,1,4.00,efectivo,tienda,Duplicado de una venta ya incluida
3001,2026-10-01,1,510,1,6.75,efectivo,tienda,Duplicado de una venta ya incluida
3006,2026-10-01,5,601,2,22.00,efectivo,tienda,Tienda no permitida
3007,2026-10-01,5,602,1,11.50,tarjeta,tienda,Tienda no permitida
3008,2026-10-01,6,701,1,16.75,efectivo,whatsapp,Tienda no permitida


## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
En este caso solo deje una copia en la GOLD, y la repetida la mande a la cuarentena. Cuando vi esos diferentes datas con el mismo ID, solo las aparté para la revisión (porque no habia una regla que dijera cual era correcta).
2. ¿Qué hicieron con un monto que venía en texto?
Aqui interprete el formato y lo pase a decimal. Los montos vacíos o no interpretables como  N/D y pendiente se mandan a cuarentena, siempre dejando el valor original.
3. ¿Por qué una tienda se quedó fuera?
Porque apliqué la regla del negocio que hace que tiendas 1, 2 y 3 sean admitidas, mande ventas de otras tiendas a cuarentena con "Tienda no permitida".
4. ¿Dónde quedó la columna que no estaba en el contrato?
Canal la deje en bronce y en registros de cuarentena, pero si la saque de orofactventas, porque no pertenece a las siete columnas.
5. Si corren el notebook otra vez, ¿el hecho crece?
No lo hace, porque para eso use el create or replace temp view, para reemplazar vistas y dejar una fila, una venta. Después de rehacer la comprobación se mantuvieron las 26 en gold y 33 en cuarentena.

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio